In [5]:
import pandas as pd
from jellyfish import jaro_winkler_similarity

# 读取数据
df = pd.read_excel("AAA.xlsx", sheet_name="系统合同主体找不到对应OA审批单")

print(df.describe())


# 相似度计算函数
def find_best_match(subject, companies):
    subject_clean = ''.join(filter(str.isalnum, str(subject).lower()))
    best_match = (None, 0)
    for idx, company in enumerate(companies):
        company_clean = ''.join(filter(str.isalnum, str(company).lower()))
        score = jaro_winkler_similarity(subject_clean, company_clean)
        if score > best_match[1]:
            best_match = (idx, score)
    return best_match[0]

# 执行匹配
subjects = df['合同主体'].dropna().tolist()
companies = df['公司名'].dropna().tolist()

match_results = []
for subject in subjects:
    match_idx = find_best_match(subject, companies)
    match_results.append((companies[match_idx], match_idx))

# 结果写入新列
df['匹配的公司名'] = pd.Series([result[0] for result in match_results] + [None]*(len(df)-len(subjects)))
df['匹配公司名序号'] = pd.Series([result[1] for result in match_results] + [None]*(len(df)-len(subjects)))

# 保存结果
df.to_excel("AAA_matched.xlsx", index=False)


         合同主体序号       公司名序号
count  55.00000  371.000000
mean   27.00000  185.000000
std    16.02082  107.242715
min     0.00000    0.000000
25%    13.50000   92.500000
50%    27.00000  185.000000
75%    40.50000  277.500000
max    54.00000  370.000000
